In [21]:
:set -XRankNTypes
:set -XScopedTypeVariables
:set -XFlexibleContexts
:set -XFlexibleInstances
:set -XMultiParamTypeClasses
:set -XDeriveFunctor
putStrLn "Setup done."

Setup done.

# ⭐ Трансформеры комонад в Haskell

Дуальная структура, практические паттерны и комбинация с трансформерами монад

> "Монады пишут программы. Комонады наблюдают за программами. Трансформеры комонад — это линзы, встроенные в контекст друг друга."


> **📦 Зависимости**
> **Пакеты:** `base`
> **Расширения:** `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `MultiParamTypeClasses` — классы с несколькими параметрами ([→](Extensions.ipynb#multiparamtypeclasses)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


❤ Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | Мотивация | Проблема комбинирования комонад |
| 2 | Категорное определение ComonadTrans | Дуальность MonadTrans |
| 3 | EnvT | Дуал ReaderT |
| 4 | StoreT | Дуал StateT, основа Lens |
| 5 | TracedT | Дуал WriterT |
| 6 | Стеки трансформеров | Наложение и ордер |
| 7 | Практические примеры | Целлюлярный автомат, UI |
| 8 | Комбинация ComonadT + MonadT | Порядок композиции |
| 9 | Связь с Adjunctions, Lens | Математическая подложка |


## 1️⃣ Мотивация

Комонада инкапсулирует один эффект наблюдения. Реальные задачи требуют нескольких эффектов одновременно.

Пример: ячейка целлюлярного автомата знает своих соседей (Store) И имеет активный сценарий (Env).

### Дуальность: Monad ↔ Comonad

| Monad | Comonad |
|-------|----------|
| `return :: a -> m a` | `extract :: w a -> a` |
| `(>>=) :: m a -> (a -> m b) -> m b` | `extend :: (w a -> b) -> w a -> w b` |
| `MonadTrans: lift :: m a -> t m a` | `ComonadTrans: lower :: t w a -> w a` |


In [22]:
-- Comonad: dual of Monad (defined manually, no comonad package needed)
class Functor w => Comonad w where
  extract   :: w a -> a
  duplicate :: w a -> w (w a)
  extend    :: (w a -> b) -> w a -> w b
  extend f  = fmap f . duplicate
  duplicate = extend id

-- ComonadTrans: dual of MonadTrans
class ComonadTrans t where
  lower :: Comonad w => t w a -> w a

putStrLn "MonadTrans:   lift  -- injects effect into stack"
putStrLn "ComonadTrans: lower -- projects context from stack"

MonadTrans:   lift  -- injects effect into stack

ComonadTrans: lower -- projects context from stack

## 2️⃣ Категорное определение ComonadTrans

Комонада — тройка (W, ε, δ): эндофунктор, ε=extract, δ=duplicate.

Трансформер t: отображение комонад в комонады с `lower`.

**Дуальные законы ComonadTrans:**
- `extract . lower = extract`
- `lower . duplicate = duplicate . lower` (натуральность)

![ComonadTrans vs MonadTrans](../diagrams/comonads/ct_diagram.svg)


In [23]:
-- Simple Identity-like comonad for testing
newtype Id a = Id { runId :: a } deriving (Functor, Show)

instance Comonad Id where
  extract (Id a) = a
  duplicate w@(Id _) = Id w

-- EnvT laws test (using Id as base comonad)
-- EnvT e w a = (e, w a)
newtype EnvT e w a = EnvT (e, w a) deriving (Functor)

instance Comonad w => Comonad (EnvT e w) where
  extract (EnvT (_, wa)) = extract wa
  duplicate w@(EnvT (e, wa)) = EnvT (e, extend (\wa' -> EnvT (e, wa')) wa)

instance ComonadTrans (EnvT e) where
  lower (EnvT (_, wa)) = wa

askEnv :: EnvT e w a -> e
askEnv (EnvT (e, _)) = e

-- Law 1: extract . lower = extract
let w1 = EnvT ("cfg", Id 42) :: EnvT String Id Int
let lhs1 = (extract . lower) w1
let rhs1 = extract w1
putStrLn $ "Law 1 (extract.lower = extract): " ++ show (lhs1 == rhs1)
putStrLn $ "extract = " ++ show lhs1

Law 1 (extract.lower = extract): True

extract = 42

## 3️⃣ EnvT — среда с фокусом

`EnvT e w a ~ (e, w a)` — добавляет неизменяющуюся среду любой комонаде.

**Дуаль**: ReaderT

| Операция | EnvT | ReaderT |
|-----------|---------|----------|
| Чтение среды | `askEnv` | `ask` |
| Чтение значения | `extract` | `runReader r` |
| Проекция | `lower` | `lift` |

**Применение**: темы UI, глобальная конфигурация


In [24]:
-- EnvT adds an immutable environment e on top of comonad w
-- (already defined in cell 2, reuse here)

-- Example: UI components with a global theme
type Theme = String
type UIComp a = EnvT Theme Id a

uiButton :: UIComp String
uiButton = EnvT ("dark", Id "<button>")

uiInput :: UIComp String
uiInput = EnvT ("dark", Id "<input>")

-- extend: apply function that uses both theme and value
renderWithTheme :: UIComp String -> String
renderWithTheme w = "[" ++ askEnv w ++ "] " ++ extract (lower w)

demoEnvT :: IO ()
demoEnvT = do
  putStrLn $ "Theme: " ++ askEnv uiButton
  putStrLn $ "Rendered button: " ++ renderWithTheme uiButton
  putStrLn $ "Rendered input:  " ++ renderWithTheme uiInput
  -- switch theme: create new EnvT with different env
  let lightButton = EnvT ("light", lower uiButton) :: UIComp String
  putStrLn $ "Light theme: " ++ renderWithTheme lightButton
demoEnvT

Theme: dark
Rendered button: [dark] <button>
Rendered input:  [dark] <input>
Light theme: [light] <button>

## 4️⃣ StoreT — хранилище с фокусом

`StoreT s w a ~ w (s -> a), s` — комонада с функцией просмотра и текущей позицией.

**Дуаль**: StateT

| Операция | StoreT | StateT |
|-----------|---------|--------|
| `pos` | текущая позиция | текущее состояние |
| `peek s` | посмотреть в позицию s | `get` |
| `seek s` | движение фокуса | `put s` |

**Основа Lens**: Store - это простейшая модель линзы в Haskell.


In [25]:
-- StoreT s w a = w (s -> a), s
-- Dual to StateT
data StoreT s w a = StoreT (w (s -> a)) s

instance Functor w => Functor (StoreT s w) where
  fmap f (StoreT wf s) = StoreT (fmap (f .) wf) s

instance Comonad w => Comonad (StoreT s w) where
  extract (StoreT wf s) = extract wf s
  duplicate (StoreT wf s) = StoreT (extend (\wf' -> \s' -> StoreT wf' s') wf) s

instance ComonadTrans (StoreT s) where
  lower (StoreT wf _) = fmap ($ undefined) wf  -- simplified

-- Core operations
pos :: StoreT s w a -> s
pos (StoreT _ s) = s

peek :: Comonad w => s -> StoreT s w a -> a
peek s (StoreT wf _) = extract wf s

seek :: s -> StoreT s w a -> StoreT s w a
seek s (StoreT wf _) = StoreT wf s

-- Simple Store (base case: Store s a = s -> a, s)
type Store s a = StoreT s Id a

mkStore :: (s -> a) -> s -> Store s a
mkStore f s = StoreT (Id f) s

-- 1D cellular automaton
type Grid1D = Store Int Bool

rule30 :: Int -> Bool
rule30 n = n `mod` 3 == 0  -- simplified rule

initialGrid :: Grid1D
initialGrid = mkStore rule30 5

demoStore :: IO ()
demoStore = do
  putStrLn $ "Pos: " ++ show (pos initialGrid)
  putStrLn $ "Cell at 5: " ++ show (peek 5 initialGrid)
  putStrLn $ "Cell at 3: " ++ show (peek 3 initialGrid)
  putStrLn $ "Cell at 6: " ++ show (peek 6 initialGrid)
demoStore

Line 10: Avoid lambda
Found:
(\ wf' -> \ s' -> StoreT wf' s')
Why not:
StoreTLine 10: Avoid lambda
Found:
\ s' -> StoreT wf' s'
Why not:
StoreT wf'Line 29: Eta reduce
Found:
mkStore f s = StoreT (Id f) s
Why not:
mkStore f = StoreT (Id f)

Pos: 5
Cell at 5: False
Cell at 3: True
Cell at 6: True

## 5️⃣ TracedT — аккумуляция с фокусом

`TracedT m w a ~ w (m -> a)` — комонада с синтезом по моноиду.

**Дуаль**: WriterT

| Операция | TracedT | WriterT |
|-----------|---------|----------|
| внедрение/чтение | `trace` | `tell`/`listen` |
| проекция | `lower` | `lift` |

**Применение**: анимация, потоки во времени


In [26]:
-- TracedT m w a = w (m -> a)  (m must be Monoid)
-- Dual to WriterT
newtype TracedT m w a = TracedT { runTracedT :: w (m -> a) }

instance Functor w => Functor (TracedT m w) where
  fmap f (TracedT w) = TracedT (fmap (f .) w)

instance (Comonad w, Monoid m) => Comonad (TracedT m w) where
  extract (TracedT w) = extract w mempty
  duplicate (TracedT w) = TracedT $ extend (\w' -> \m -> TracedT $ fmap (. (m <>)) w') w

instance Monoid m => ComonadTrans (TracedT m) where
  lower (TracedT w) = fmap ($ mempty) w

-- trace: read value at a specific monoidal position
traceAt :: Comonad w => m -> TracedT m w a -> a
traceAt m (TracedT w) = extract w m

-- Example: Fibonacci as a Traced comonad
type Traced' m a = TracedT m Id a

fibs :: Traced' (Sum Int) Int
fibs = TracedT $ Id $ \(Sum n) ->
  let go 0 = 0
      go 1 = 1
      go k = go (k-1) + go (k-2)
  in go n

newtype Sum a = Sum { getSum :: a } deriving (Show)
instance Num a => Semigroup (Sum a) where Sum a <> Sum b = Sum (a + b)
instance Num a => Monoid (Sum a) where mempty = Sum 0

demoTraced :: IO ()
demoTraced = do
  putStrLn $ "fib(0..9): " ++ show [traceAt (Sum i) fibs | i <- [0..9]]
  -- derivative: consecutive differences
  let diff = extend (\w -> traceAt (Sum 1) w - extract w) fibs
  putStrLn $ "diff(1..9): " ++ show [traceAt (Sum i) diff | i <- [1..9]]
demoTraced

Line 10: Collapse lambdas
Found:
\ w' -> \ m -> TracedT $ fmap (. (m <>)) w'
Why not:
\ w' m -> TracedT $ fmap (. (m <>)) w'

fib(0..9): [0,1,1,2,3,5,8,13,21,34]
diff(1..9): [0,1,1,2,3,5,8,13,21]

## 6️⃣ Стеки трансформеров комонад

Трансформеры можно складывать друг на друга:

```
EnvT Scenario (Store Pos) Bool
-- Outer (global): Env provides Scenario
-- Inner (local):  Store provides Pos focus
```

**Правило**: `lower` = проекция, порядок влияет на семантику.

![Stacks of comonad transformers](../diagrams/comonads/ct_stack.svg)


In [27]:
-- Stack: EnvT Scenario (Store Pos) Bool
-- Outer (global): environment/scenario
-- Inner (local):  position/focus
type Scenario = String
type Pos = Int
type CAWithScenario = EnvT Scenario (StoreT Pos Id) Bool

-- Create a cellular automaton with a scenario
makeCellAuto :: Scenario -> [Bool] -> Pos -> CAWithScenario
makeCellAuto scenario cells p =
  let lookup i = if i >= 0 && i < length cells
                 then cells !! i
                 else False
  in EnvT (scenario, StoreT (Id lookup) p)

demoStack :: IO ()
demoStack = do
  let ca = makeCellAuto "rule-30" [True,False,True,True,False] 2
  putStrLn $ "Scenario: " ++ askEnv ca
  putStrLn $ "Cell at focus: " ++ show (extract ca)
  -- lower gives us Store Pos Bool (without scenario)
  let innerStore = lower ca
  putStrLn $ "Inner store pos: " ++ show (pos innerStore)
  putStrLn $ "Cell at 0: " ++ show (peek 0 innerStore)
  putStrLn $ "Cell at 1: " ++ show (peek 1 innerStore)
demoStack

Line 11: Redundant if
Found:
if i >= 0 && i < length cells then cells !! i else False
Why not:
((i >= 0 && i < length cells) && (cells !! i))

Scenario: rule-30
Cell at focus: True
Inner store pos: 2
Cell at 0: True
Cell at 1: False

## 7️⃣ Практические примеры

Целлюлярный автомат через Store:

- `Store (Int,Int) Bool` — бесконечная сетка
- `extend rule grid` — один шаг эволюции
- `extract` — значение в фокусе

**Ключевой инсайт**: `extend f` = применить `f` ко всему контексту одновременно.


In [28]:
-- 2D cellular automaton with StoreT
type Grid = StoreT (Int,Int) Id Bool

-- Create a grid from a list of lists
makeGrid :: [[Bool]] -> (Int,Int) -> Grid
makeGrid cells focus =
  let r = length cells
      c = if r > 0 then length (cells !! 0) else 0
      lookup (row,col) =
        if row >= 0 && row < r && col >= 0 && col < c
        then (cells !! row) !! col
        else False
  in StoreT (Id lookup) focus

-- Count live neighbors
countNeighbors :: Grid -> Int
countNeighbors g = length $ filter id
  [ peek (row+dr, col+dc) g
  | dr <- [-1,0,1], dc <- [-1,0,1]
  , not (dr == 0 && dc == 0)
  ]
  where (row,col) = pos g

-- Conway's Game of Life rule
golStep :: Grid -> Bool
golStep g =
  let alive = extract g
      n     = countNeighbors g
  in case (alive, n) of
       (True,  2) -> True
       (True,  3) -> True
       (False, 3) -> True
       _          -> False

-- Glider pattern
glider :: [[Bool]]
glider = [ [False,True,False]
         , [False,False,True]
         , [True, True, True] ]

demoGol :: IO ()
demoGol = do
  let g  = makeGrid glider (1,1)
  putStrLn $ "Cell (1,1): " ++ show (extract g)
  putStrLn $ "Neighbors of (1,1): " ++ show (countNeighbors g)
  let g2 = extend golStep g
  putStrLn $ "After 1 step (1,1): " ++ show (extract g2)
demoGol

Line 8: Use head
Found:
cells !! 0
Why not:
head cellsLine 10: Redundant if
Found:
if row >= 0 && row < r && col >= 0 && col < c then
    (cells !! row) !! col
else
    False
Why not:
((row >= 0 && row < r && col >= 0 && col < c)
   && ((cells !! row) !! col))

Cell (1,1): False
Neighbors of (1,1): 5
After 1 step (1,1): False

## 8️⃣ Комбинация ComonadT + MonadT

Комонады и монады можно комбинировать двумя способами.

### Паттерн 1: ComonadT поверх Monad (`EnvT Config IO`)

```
EnvT Config IO a
-- IO внутри, EnvT снаружи
-- extract :: IO невозможен (IO не комонада)!
-- Работает только через специальный интерфейс
```

### Паттерн 2: MonadT поверх Comonad (`StateT s (Store s)`)

```
StateT s (Store s) a
-- Store комонада внутри
-- StateT монада снаружи
-- Комонада обеспечивает READ (observe surroundings)
-- Монада обеспечивает WRITE (update state)
```

### Порядок композиции

| Порядок | Тип | Семантика |
|--------|-----|----------|
| `ComonadT (MonadT m) w` | глобальный контекст + локальные эффекты | `EnvT Config (StateT s Identity)` |
| `MonadT m (ComonadT t w)` | локальные эффекты + контекстное наблюдение | `StateT s (Store s)` |

**Основное правило**: внешний = глобальный, внутренний = локальный.


In [29]:
-- Pattern 1: ComonadT over a Monad
-- EnvT Config IO: IO inside, EnvT outside
-- We have a fixed Config AND can do IO effects
type Config = String

explainCombo :: IO ()
explainCombo = do
  putStrLn "EnvT Config IO a:"
  putStrLn "  extract: get IO action (runs effects)"
  putStrLn "  askEnv:  read config (comonadic)"
  putStrLn ""
  putStrLn "StateT s (Store s) a:"
  putStrLn "  Store provides READ context (observe surroundings)"
  putStrLn "  State provides WRITE effects (update state, do IO)"
  putStrLn ""
  putStrLn "Order: outer = global, inner = local"
explainCombo

-- Pattern 2: MonadT over a Comonad (conceptual demo)
import Control.Monad.Trans.State (StateT, runStateT, get, put)

-- Store-based grid with stateful navigation
-- StateT Int (StoreT Int Id) Bool
-- Store provides: read cell value at any position
-- State provides: track/update current position

-- Demo: navigate a 1D grid using State for position tracking
navDemo :: IO ()
navDemo = do
  putStrLn "Pattern 2: StateT over StoreT"
  let grid = StoreT (Id (\p -> p `elem` [1,3,5])) (0 :: Int)  -- odd positions live
  -- Run a stateful navigation on the grid
  let navAction = do
        put (3 :: Int)   -- navigate to position 3
        s <- get
        return s
  let (finalPos, _) = extract (runStateT navAction 0)
  putStrLn $ "Navigated to: " ++ show finalPos
  putStrLn $ "Cell alive: " ++ show (peek finalPos grid)
navDemo

EnvT Config IO a:
  extract: get IO action (runs effects)
  askEnv:  read config (comonadic)

StateT s (Store s) a:
  Store provides READ context (observe surroundings)
  State provides WRITE effects (update state, do IO)

Order: outer = global, inner = local

: 

## 9️⃣ Связь с Adjunctions, Lens

### Сопряжения (-|) и трансформеры

Если `F -| G` (функтор `F` левый смежный с `G`):

- `F` порождает монаду: `return = unit`, `(>>=) = counit`
- `G` порождает комонаду: `extract = counit`, `duplicate = unit`

### Store и Lens

Store s a ≡ (s -> a, s) — это именно определение Lens в van Laarhoven!

```
type Lens s a = s -> Store a s
-- эквивалентно:
type Lens s a = forall f. Functor f => (a -> f a) -> s -> f s
```

**Store ↔ State**: сопряжение `State s -| Store s`.


In [30]:
-- Store s a = (s -> a, s): this IS the definition of a Lens!
-- Lens: s -> Store a s  (focus on 'a' inside 's')
-- Store a s = StoreT a Id s: position=a, value=s

-- Simple lens type using Store
type Lens' s a = s -> StoreT a Id s

-- Lens for first element of a pair
fstLens :: Lens' (a, b) a
fstLens (x, y) = StoreT (Id (\x' -> (x', y))) x

-- view: get the focused value
viewLens :: Lens' s a -> s -> a
viewLens ln s = pos (ln s)

-- set: set the focused value  
setLens :: Lens' s a -> a -> s -> s
setLens ln a s = peek a (ln s)

-- over: modify the focused value
overLens :: Lens' s a -> (a -> a) -> s -> s
overLens ln f s = setLens ln (f (viewLens ln s)) s

demonstrateLens :: IO ()
demonstrateLens = do
  let pair = (42 :: Int, "hello")
  putStrLn $ "Focus (fst): " ++ show (viewLens fstLens pair)
  putStrLn $ "Set to 100: " ++ show (setLens fstLens 100 pair)
  putStrLn $ "Add 1: " ++ show (overLens fstLens (+1) pair)
  -- extend: apply to all positions comonadically
  let store = fstLens pair  -- StoreT focused on 42
  let store' = extend (\w -> peek (pos w + 1) w) store
  putStrLn $ "After +1 (fst): " ++ show (pos store')
demonstrateLens

Line 10: Use tuple-section
Found:
\ x' -> (x', y)
Why not:
(, y)

Focus (fst): 42
Set to 100: (100,"hello")
Add 1: (43,"hello")
After +1 (fst): 42

## 📊 Итог

| Трансформер | Тип | Дуаль | Применение |
|-----------|------|-------|------------|
| EnvT e w a | `(e, w a)` | ReaderT | Темы, конфигурация |
| StoreT s w a | `w(s->a), s` | StateT | Lens, ЦА |
| TracedT m w a | `w(m->a)` | WriterT | Анимация, потоки |
| ComonadT+MonadT | смешанный | — | UI, gameplay |

**Ключевое правило**: `lower` = проекция, `lift` = въем. Внешний = глобальный, внутренний = локальный.


---

**← Предыдущий:** [Комонады](Comonads.ipynb)  |  **Следующий →** [Foldable & Traversable](FoldableTraversable.ipynb)
